<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/MMD/notebooks/three_layer_agentic_refinement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title requirements
!pip install transformers datasets accelerate evaluate rouge_score torch jsonlines rake_nltk pytorch_lightning
!pip install rapidfuzz
!pip install boto3
!pip install mistralai
!pip install --upgrade mistralai

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Clone the Repo

In [5]:
# @markdown check the check-box to clone the repo from sorce, <b>otherwise it will be loaded from shared google drive files</b>

clone_repo = False # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"


# Preparing Data

In [6]:
# @title Download requirements for data preparing

%cd {path}

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

/content/drive/MyDrive/DNLP-storage/SigExt


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
# @title Run the `prepare_data.py` script
# @markdown Here you can select the dataset you want to use for the training. </br> If you want to use a new dataset, make sure to also change the `path_to_dataset`. </br> No new file will be downloaded or processed if `path_to_dataset` already exists!!

import os
dataset = "cnn" #@param{type:"string"}
path_to_dataset = "experiments/cnn_dataset/" #@param{type:"string"}
if not os.path.exists(path+"/"+path_to_dataset):
  print('New dataset...')
  !python3 src/prepare_data.py --dataset {dataset} --output_dir {path_to_dataset}
  pass

else:
  print(f"Directory already exist. `{path_to_dataset}` will be used")

Directory already exist. `experiments/cnn_dataset/` will be used


# Train

You can modify path to check-points and outputs to load or save different files.
**To keep track of procedure and training routh, insert paths you create in this text down below, under the `paths` ( DO NOT FORGET THE DESCRIPTION :) thank you in advance.  )**



 **⚠️Even for test runs, Change two paths to avoid overwriting existing results⚠️**


---


**Paths**
*   `experiments/cnn_extractor_model/` Baseline code (dataset: cnn),  no modification. output in `experiments/cnn_dataset_with_keyphrase/`
*   `new_path` add you new path here...





In [9]:

path_to_checkpoint = "experiments/cnn_extractor_model/" #@param{type:"string"}
path_to_output = "experiments/cnn_dataset_with_keyphrase/" #@param{type:"string"}

train_longformer_extractor = False #@param{type:"boolean"}
inference_longformer_extractor = False #@param{type:"boolean"}


In [10]:
#@title Calling `train_longformer_extractor_context.py` to train the longformer
if train_longformer_extractor:
  !python3 src/train_longformer_extractor_context.py \
    --dataset_dir {path_to_dataset} \
    --checkpoint_dir {path_to_checkpoint}

In [11]:
# #@title Calling `inference_longformer_extractor.py`
if inference_longformer_extractor:
  !python3 src/inference_longformer_extractor.py \
    --dataset_dir {path_to_dataset} \
    --checkpoint_dir {path_to_checkpoint} \
    --output_dir {path_to_output}

In [12]:
summerizing_result_path = "experiments/cnn_extsig_predictions_mistral-small-2603_k15_extention2/" #@param{type:"string"}
top_k_kp = 15 #@param{type:"integer"}

In [13]:
#@title PROMPTS HERE
CRITIC_PROMPT_TEMPLATE = """<s>[INST]You are a rigorous auditor. Here is the original task, source text, and required key phrases:
<original_task>
{original_task}
</original_task>

Here is the drafted summary:
<summary>
{draft_summary}
</summary>

Please compare the draft against the source text. Evaluate if the draft accurately captures the context and incorporates the key phrases. Provide a Critique Report highlighting factual discrepancies, missing metadata, or omitted key phrases.[/INST]"""

ENHANCER_PROMPT_TEMPLATE = """<s>[INST]Here is the original task, source text, and required key phrases:
<original_task>
{original_task}
</original_task>

Here is the initial draft:
<summary>
{draft_summary}
</summary>

Here is the Critique Report of that draft:
<critique>
{critique}
</critique>

Please re-write and adjust the summary. Address the flaws noted by the Critic and ensure the key phrases are accurately integrated. Output ONLY the final, verified summary without conversational filler.[/INST]"""

In [ ]:
import sys
import os
from types import ModuleType
import time
from mistralai.client import Mistral
def chat(prompt_text, client):
    chat_response = client.chat.complete(
        model="mistral-small-latest", # Recommended to use 'latest' tags for Mistral API
        messages=[{"role": "user", "content": prompt_text}]
    )
    return chat_response.choices[0].message.content

def predict_one_eg_mistral_3_layer(prompt_dict, is_3_layer_active=True):
    client = Mistral(api_key="YOUR_API_KEY_HERE")
    original_task = prompt_dict["prompt_input"]

    # Layer 1: Base Extraction
    draft_summary = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            if is_3_layer_active:
                draft_summary = chat(original_task, client)
            else:
                final_output = chat(original_task, client)
            break
        except Exception as e:
            if "429" in str(e):
                print("L1 Rate limit hit! Sleeping for 10s...")
                time.sleep(10)
    if not is_3_layer_active:
        return final_output or "Error: Layer 3 Failed"

    if not draft_summary: return "Error: Layer 1 Failed"

    # Layer 2: Critic Verification
    critic_prompt = CRITIC_PROMPT_TEMPLATE.format(original_task=original_task, draft_summary=draft_summary)
    critique = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            critique = chat(critic_prompt, client)
            break
        except Exception as e:
            if "429" in str(e):
                print("L2 Rate limit hit! Sleeping for 10s...")
                time.sleep(10)

    if not critique: return "Error: Layer 2 Failed"

    # Layer 3: Enhancement & Polish
    enhancer_prompt = ENHANCER_PROMPT_TEMPLATE.format(original_task=original_task, draft_summary=draft_summary, critique=critique)
    final_output = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            final_output = chat(enhancer_prompt, client)
            break
        except Exception as e:
            if "429" in str(e):
                print("L3 Rate limit hit! Sleeping for 10s...")
                time.sleep(10)

    return final_output or "Error: Layer 3 Failed"

class FakePool:
    def __init__(self, processes=None): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
    def imap(self, func, iterable):
        return map(func, iterable)


import multiprocessing

multiprocessing.Pool = FakePool


m = ModuleType("bedrock_utils")
m.predict_one_eg_mistral = predict_one_eg_mistral_3_layer
m.predict_one_eg_claude_instant = lambda x: "Not configured"
sys.modules["bedrock_utils"] = m


sys.argv = [
    "zs_summarization.py",
    "--model_name", "mistral",
    "--kw_strategy", "sigext_topk",
    "--kw_model_top_k", str(top_k_kp),
    "--dataset", dataset,
    "--dataset_dir", path_to_output,
    "--output_dir", summerizing_result_path
]



sys.path.append('/content/drive/MyDrive/DNLP-storage/SigExt/src')
import zs_summarization
import importlib
importlib.reload(zs_summarization)

zs_summarization.main()

100%|██████████| 500/500 [55:05<00:00,  6.61s/it]


In [14]:
import jsonlines
base = "/content/drive/MyDrive/DNLP-storage/SigExt/experiments"
path_predictions      = f"{base}/experiments/cnn_extsig_predictions_mistral-small-2603_k15_extention2/test_predictions.json" #Path to the 500 raw summaries generated by Mistral
path_full_dataset     = f"{base}/experiments/cnn_extsig_predictions_mistral-small-2603_k15_extention2/test_dataset.jsonl" #Path to the full dataset file that includes raw_output

with jsonlines.open(path_full_dataset) as f:
  full_dataset = list(f)
with jsonlines.open(path_full_dataset) as f:
  full_dataset = list(f)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DNLP-storage/SigExt/src/experiments/cnn_extsig_predictions_mistral-small-2603_k15_extention2/test_dataset.jsonl'

In [ ]:
#@title  Evaluate and Save Results
from rouge_score import rouge_scorer
import numpy as np
import json
import os

def compute_rouge(predictions, references):
    scorer_obj = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"], use_stemmer=True
    )
    r1_f, rl_f, r1_r = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not pred.strip():
            pred = "empty"
        score = scorer_obj.score(target=ref, prediction=pred)
        r1_f.append(score["rouge1"].fmeasure)
        rl_f.append(score["rougeL"].fmeasure)
        r1_r.append(score["rouge1"].recall)
    return {
        "rouge1f": round(np.mean(r1_f) * 100, 4),
        "rougeLf": round(np.mean(rl_f) * 100, 4),
        "rouge1r": round(np.mean(r1_r) * 100, 4),
    }

# human reference summaries
references = [item["raw_output"] for item in full_dataset]

# compute ROUGE for raw and verified summaries
raw_metrics   = compute_rouge(raw_summaries,   references)
final_metrics = compute_rouge(final_summaries, references)

# hallucination statistics
total_sentences = sum(
    len(split_sentences(log["raw_summary"]))
    for log in all_halluc_logs
    if log["raw_summary"]
)
total_flagged = sum(
    len(log["hallucinations"])
    for log in all_halluc_logs
)
regen_accepted = sum(
    sum(1 for h in log["hallucinations"] if h["regen_accepted"])
    for log in all_halluc_logs
)
dropped      = total_flagged - regen_accepted
halluc_rate  = total_flagged / total_sentences if total_sentences > 0 else 0

# print results table
print("=" * 52)
print(f"{'Metric':<28} {'Baseline':>10} {'Verified':>10}")
print("=" * 52)
print(f"{'ROUGE-1 F1':<28} {raw_metrics['rouge1f']:>10} {final_metrics['rouge1f']:>10}")
print(f"{'ROUGE-L F1':<28} {raw_metrics['rougeLf']:>10} {final_metrics['rougeLf']:>10}")
print(f"{'ROUGE-1 Recall':<28} {raw_metrics['rouge1r']:>10} {final_metrics['rouge1r']:>10}")
print("=" * 52)
print(f"{'Total sentences':<28} {total_sentences:>21}")
print(f"{'Flagged sentences':<28} {total_flagged:>21}")
print(f"{'Hallucination rate':<28} {halluc_rate:>20.2%}")
print(f"{'Regenerated accepted':<28} {regen_accepted:>21}")
print(f"{'Dropped':<28} {dropped:>21}")
print("=" * 52)

# save results to Drive
results = {
    "raw_metrics":        raw_metrics,
    "verified_metrics":   final_metrics,
    "hallucination_rate": round(halluc_rate, 4),
    "total_sentences":    total_sentences,
    "total_flagged":      total_flagged,
    "regen_accepted":     regen_accepted,
    "dropped":            dropped,
}

os.makedirs(path_output, exist_ok=True)

with open(f"{path_output}metrics_verified.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nmetrics_verified.json saved")

with open(f"{path_output}halluc_log.json", "w") as f:
    json.dump(all_halluc_logs, f, indent=2)
print("halluc_log.json saved")

with open(f"{path_output}final_predictions.json", "w") as f:
    json.dump(final_summaries, f, indent=2)
print("final_predictions.json saved")

# verify files exist and show sizes
print("\nFiles saved:")
for fname in ["metrics_verified.json", "halluc_log.json", "final_predictions.json"]:
    fpath = f"{path_output}{fname}"
    size  = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    print(f"  {fname}  ({size:,} bytes)")